In [0]:
#Leitura camada bronze
df_conta = spark.table("bronze.conta_diaria_05hs")

In [0]:
from pyspark.sql import functions as F

df_silver_conta = (
  df_conta
  .select(
    F.col("id").cast("long").alias("id"),
    F.initcap(F.col("tipo")).alias("tipo"),  # Padroniza tipo com inicial maiúscula
    F.col("data_criacao_conta").cast("date").alias("data_criacao_conta"),
    F.col("id_associado").cast("long").alias("id_associado"),
    F.current_timestamp().alias("dt_ingestao_silver")
  )
  .dropDuplicates(["id"])
)
#display(df_silver_conta)

In [0]:
# Qualidade dos dados
erros = []  # Lista para armazenar erros de qualidade identificados

# Verifica se existe algum registro com id nulo
if df_silver_conta.filter(F.col("id").isNull()).count() > 0: 
    erros.append("id nulo")

# Verifica se existe algum registro com tipo nulo ou vazio
if df_silver_conta.filter(F.col("tipo").isNull() | (F.col("tipo") == "")).count() > 0: 
    erros.append("tipo nulo ou vazio")

# Verifica se existe algum registro com data_criacao_conta nula
if df_silver_conta.filter(F.col("data_criacao_conta").isNull()).count() > 0: 
    erros.append("data_criacao_conta nula")

# Verifica se existe algum registro com id_associado nulo
if df_silver_conta.filter(F.col("id_associado").isNull()).count() > 0: 
    erros.append("id_associado nulo")

# Se houver erros, imprime cada erro e lança uma exceção caso necessario
if erros:
    for erro in erros:
        print(f"Falha qualidade conta_silver: {erro}")
    raise Exception("Falha qualidade conta_silver: " + " | ".join(erros))

In [0]:
#Salvando base na camada silver e modo overwrite
(df_silver_conta.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("silver.conta_diaria_05hs"))